# Übungen – Block 2: Linux-Grundlagen II

Dieses Notebook enthält die praktischen Übungen zu **Administration und einfacher Automatisierung**.

## Lernziele

Sie üben:

- Paketmanagement,
- Prozesse und Dienste,
- Netzwerkgrundlagen,
- SSH,
- SSH-Keys,
- einfache Shell-Skripte,
- Exit Codes und einfache Bedingungen,
- Orchestrierung mehrerer vorhandener Programme.

> Einige Aufgaben benötigen `sudo`, `systemd`, Netzwerkzugriff oder einen vorbereiteten SSH-Zielhost. Führen Sie diese bei Bedarf im separaten Schulungs-Terminal aus.


## Musterlösung

Die folgenden Lösungen zeigen bewusst einfache Varianten. Bei Shell- und Linux-Aufgaben sind häufig mehrere korrekte Lösungswege möglich.

Für `sudo`, `systemctl`, Paketinstallation und SSH müssen Befehle gegebenenfalls im separaten Schulungs-Terminal ausgeführt und an die konkrete Umgebung angepasst werden.


## Vorbereitung

Die folgende Zelle erzeugt lokale Dateien für die Shell-Scripting-Übungen.


In [ ]:
%%bash
set -e
BASE=/tmp/linux_block2
rm -rf "$BASE"
mkdir -p "$BASE"/{scripts,data,logs}

cat > "$BASE/data/check.txt" <<'EOF'
server_endpoint=localhost
application=demo
EOF

echo "Übungsumgebung: $BASE"
find "$BASE" -maxdepth 2 -print


---

# Übung 1: Paketmanagement

## Aufgabe

Verwenden Sie den Paketmanager Ihrer Schulungsumgebung.

1. Ermitteln Sie, welcher Paketmanager verfügbar ist.
2. Suchen Sie nach einem vom Trainer vorgegebenen kleinen Paket.
3. Zeigen Sie Paketinformationen an.
4. Installieren Sie das Paket.
5. Prüfen Sie, ob das zugehörige Programm ausgeführt werden kann.
6. Entfernen Sie das Paket wieder.

### Denkfragen

- Was bewirkt bei `apt` ein `apt update`?
- Warum können bei einer Paketinstallation zusätzliche Pakete installiert werden?


In [ ]:
%%bash
# Beispiel für Debian/Ubuntu:
# sudo apt update
# apt search tree
# apt show tree
# sudo apt install tree
# tree --version
# sudo apt remove tree

echo "Musterlösung: Befehle an Paketmanager und Schulungsumgebung anpassen."


---

# Übung 2: Prozesse untersuchen und beenden

## Aufgabe

1. Starten Sie im Hintergrund einen ungefährlichen Prozess, der längere Zeit läuft.
2. Ermitteln Sie dessen PID.
3. Finden Sie den Prozess mit `ps` oder `pgrep`.
4. Beenden Sie ihn regulär mit `kill`.
5. Prüfen Sie anschließend, ob der Prozess noch läuft.

Verwenden Sie keine fremden oder systemkritischen Prozesse.


In [ ]:
%%bash
sleep 300 &
PID=$!

echo "Gestartete PID: $PID"
ps -p "$PID" -f

kill "$PID"
sleep 1

if ps -p "$PID" > /dev/null; then
    echo "Prozess läuft noch"
else
    echo "Prozess wurde beendet"
fi


---

# Übung 3: Dienste mit systemd

Verwenden Sie einen vom Trainer freigegebenen Dienst.

## Aufgabe

1. Prüfen Sie den Status des Dienstes.
2. Ermitteln Sie, ob der Dienst aktuell aktiv ist.
3. Ermitteln Sie, ob er für den Systemstart aktiviert ist.
4. Starten oder stoppen Sie ihn gemäß Traineranweisung.
5. Starten Sie ihn anschließend neu.
6. Prüfen Sie erneut Status und Aktivierungszustand.

### Denkfrage

Was ist der Unterschied zwischen `start` und `enable`?


In [ ]:
%%bash
# Beispiel mit einem vom Trainer freigegebenen Dienst:
SERVICE=ssh

systemctl status "$SERVICE" --no-pager || true
systemctl is-active "$SERVICE" || true
systemctl is-enabled "$SERVICE" || true

# Administrative Änderungen ggf. im Terminal:
# sudo systemctl restart "$SERVICE"

echo
echo "Musterantwort:"
echo "start startet einen Dienst jetzt; enable aktiviert seinen Start beim Systemboot."


---

# Übung 4: Netzwerkgrundlagen

## Aufgabe

1. Ermitteln Sie den Hostnamen des Systems.
2. Ermitteln Sie die lokalen IP-Adressen.
3. Zeigen Sie die Routing-Tabelle bzw. Standardroute an.
4. Prüfen Sie die Erreichbarkeit eines vom Trainer vorgegebenen Zielsystems.
5. Prüfen Sie optional, welche TCP-Ports lokal lauschen.

### Denkfrage

Warum beweist ein erfolgreiches `ping` nicht, dass beispielsweise ein Web- oder SSH-Dienst verfügbar ist?


In [ ]:
%%bash
echo "Hostname:"
hostname

echo
echo "IP-Adressen:"
hostname -I || true

echo
echo "Interfaces:"
ip addr || true

echo
echo "Routing:"
ip route || true

echo
echo "Ping localhost:"
ping -c 1 127.0.0.1 || true

echo
echo "Lauschende TCP-Ports:"
ss -ltn 2>/dev/null || true


---

# Übung 5: Remote-Zugriff mit SSH

Für diese Übung benötigen Sie die vom Trainer bereitgestellten Zugangsdaten für einen Zielhost.

## Aufgabe

1. Verbinden Sie sich mit dem Zielsystem per SSH.
2. Ermitteln Sie dort:
   - den aktuellen Benutzer,
   - den Hostnamen,
   - das aktuelle Arbeitsverzeichnis.
3. Beenden Sie die Remote-Sitzung.
4. Führen Sie anschließend einen einzelnen Remote-Befehl direkt über `ssh` aus, ohne eine interaktive Sitzung geöffnet zu lassen.

### Kontrollfrage

Woran erkennen Sie sicher, ob ein Kommando lokal oder auf dem Remote-System ausgeführt wurde?


In [ ]:
%%bash
# An Schulungsumgebung anpassen:
# ssh user@host
# Auf dem Remote-System:
# whoami
# hostname
# pwd
# exit
#
# Einzelner Remote-Befehl:
# ssh user@host 'hostname && whoami'

echo "SSH-Zugangsdaten werden vom Trainer vorgegeben."


---

# Übung 6: SSH-Key-Authentifizierung

## Aufgabe

1. Erzeugen Sie ein neues SSH-Schlüsselpaar ausschließlich für die Schulungsumgebung.
2. Prüfen Sie, welche beiden Dateien entstanden sind.
3. Identifizieren Sie privaten und öffentlichen Schlüssel.
4. Übertragen Sie **nur den öffentlichen Schlüssel** auf den bereitgestellten Zielhost.
5. Testen Sie die Anmeldung mit dem Schlüssel.

### Denkfragen

- Welche Datei darf niemals an den Zielserver weitergegeben werden?
- Wozu dient `authorized_keys` auf dem Zielsystem?


In [ ]:
%%bash
# Beispiel:
# ssh-keygen -t ed25519 -f ~/.ssh/linux_training -C "linux-training"
# ls -l ~/.ssh/linux_training*
# ssh-copy-id -i ~/.ssh/linux_training.pub user@host
# ssh -i ~/.ssh/linux_training user@host
#
# Der private Schlüssel ist ~/.ssh/linux_training
# Der öffentliche Schlüssel ist ~/.ssh/linux_training.pub

echo "Nur der öffentliche Schlüssel wird auf dem Zielsystem hinterlegt."


---

# Übung 7: Erstes Shell-Skript

Erstellen Sie unter `/tmp/linux_block2/scripts` ein Skript `systemcheck.sh`.

## Anforderungen

Das Skript soll:

1. eine Startmeldung ausgeben,
2. den Hostnamen in einer Variablen speichern,
3. den Hostnamen ausgeben,
4. das aktuelle Datum ausgeben,
5. die Datei `/tmp/linux_block2/data/check.txt` anzeigen,
6. eine Abschlussmeldung ausgeben.

Machen Sie das Skript ausführbar und starten Sie es direkt.


In [ ]:
%%bash
BASE=/tmp/linux_block2
SCRIPT="$BASE/scripts/systemcheck.sh"

cat > "$SCRIPT" <<'EOF'
#!/bin/bash

echo "Systemcheck startet"

HOST=$(hostname)
echo "Hostname: $HOST"

date

echo "Konfiguration:"
cat /tmp/linux_block2/data/check.txt

echo "Systemcheck beendet"
EOF

chmod +x "$SCRIPT"
"$SCRIPT"


---

# Übung 8: Exit Codes

## Aufgabe

1. Führen Sie ein Kommando aus, das erfolgreich endet.
2. Geben Sie unmittelbar danach dessen Exit Code aus.
3. Führen Sie ein Kommando aus, das fehlschlägt.
4. Geben Sie unmittelbar danach erneut den Exit Code aus.
5. Ergänzen Sie `systemcheck.sh` so, dass ein Prüfschritt ausgeführt wird und der Exit Code ausgewertet wird.

### Ziel

Bei Erfolg soll das Skript `Prüfung erfolgreich` ausgeben, bei einem Fehler `Prüfung fehlgeschlagen`.


In [ ]:
%%bash
echo "Erfolgreiches Kommando:"
true
echo "Exit Code: $?"

echo
echo "Fehlschlagendes Kommando:"
false
echo "Exit Code: $?"

echo
echo "Auswertung in einer Bedingung:"
if test -f /tmp/linux_block2/data/check.txt; then
    echo "Prüfung erfolgreich"
else
    echo "Prüfung fehlgeschlagen"
fi


---

# Übung 9: Folgeschritte nur bei Erfolg

Erstellen Sie ein Skript `pipeline.sh`.

## Anforderungen

Das Skript soll:

1. einen ersten Prüfschritt ausführen,
2. nur bei Erfolg einen zweiten Schritt ausführen,
3. bei einem Fehler eine verständliche Meldung ausgeben,
4. einen Status in `/tmp/linux_block2/logs/pipeline.log` schreiben.

Lösen Sie die Aufgabe zunächst mit `&&` und danach mit einer einfachen `if`-Bedingung.


In [ ]:
%%bash
BASE=/tmp/linux_block2
SCRIPT="$BASE/scripts/pipeline.sh"

cat > "$SCRIPT" <<'EOF'
#!/bin/bash

FILE=/tmp/linux_block2/data/check.txt
LOG=/tmp/linux_block2/logs/pipeline.log

# Variante mit &&
test -f "$FILE" && echo "Datei vorhanden"

# Variante mit if
if grep -q "server_endpoint" "$FILE"; then
    echo "Konfiguration geprüft"
    echo "pipeline: erfolgreich" >> "$LOG"
else
    echo "Konfiguration fehlerhaft"
    echo "pipeline: fehlgeschlagen" >> "$LOG"
    exit 1
fi
EOF

chmod +x "$SCRIPT"
"$SCRIPT"

echo
cat "$BASE/logs/pipeline.log"


---

# Übung 10: Einfache Orchestrierung

Erstellen Sie ein Shell-Skript, das mehrere vorhandene Werkzeuge zu einem Ablauf verbindet.

## Anforderungen

Das Skript soll nacheinander:

1. prüfen, ob eine bestimmte Datei vorhanden ist,
2. deren Inhalt ausgeben,
3. einen Netzwerk-Endpunkt mit einem geeigneten Kommando prüfen,
4. nur bei erfolgreicher Prüfung einen abschließenden Schritt ausführen,
5. bei einem Fehler mit einer verständlichen Meldung abbrechen.

Die konkrete Programmlogik soll bewusst einfach bleiben. Ziel ist die Orchestrierung vorhandener Werkzeuge.


In [ ]:
%%bash
BASE=/tmp/linux_block2
SCRIPT="$BASE/scripts/orchestrate.sh"

cat > "$SCRIPT" <<'EOF'
#!/bin/bash

FILE=/tmp/linux_block2/data/check.txt

if ! test -f "$FILE"; then
    echo "Fehler: Konfigurationsdatei fehlt"
    exit 1
fi

cat "$FILE"

if ping -c 1 127.0.0.1 > /dev/null 2>&1; then
    echo "Netzwerkprüfung erfolgreich"
else
    echo "Netzwerkprüfung fehlgeschlagen"
    exit 1
fi

echo "Alle Schritte erfolgreich abgeschlossen"
EOF

chmod +x "$SCRIPT"
"$SCRIPT"


---

# Abschlussübung: Administrationsablauf

Entwickeln Sie einen kleinen, reproduzierbaren Administrationsablauf.

## Anforderungen

1. Ermitteln Sie Hostname und IP-Adresse des lokalen Systems.
2. Prüfen Sie einen vom Trainer vorgegebenen Dienst.
3. Prüfen Sie die Erreichbarkeit eines Zielhosts.
4. Stellen Sie per SSH eine Verbindung zu diesem Zielhost her.
5. Führen Sie dort einen ungefährlichen Prüf-Befehl aus.
6. Erstellen Sie anschließend ein Shell-Skript, das mindestens drei dieser Prüfschritte automatisiert.
7. Werten Sie mindestens einen Exit Code aus.
8. Ein weiterer Schritt darf nur dann laufen, wenn der vorherige erfolgreich war.
9. Schreiben Sie eine kurze Ergebnisinformation in eine Logdatei.

## Reflexion

Ordnen Sie anschließend die Aufgaben ein:

- Welche Schritte waren reine Linux-Administration?
- Welche Aufgabe hat die Shell im Skript übernommen?
- Wo würde bei komplexerer eigener Logik später Python sinnvoll werden?
- Welche dieser Aufgaben könnten auf vielen Hosts später mit Ansible automatisiert werden?


In [ ]:
%%bash
BASE=/tmp/linux_block2
SCRIPT="$BASE/scripts/admin_check.sh"
LOG="$BASE/logs/admin_check.log"

cat > "$SCRIPT" <<'EOF'
#!/bin/bash

LOG=/tmp/linux_block2/logs/admin_check.log

echo "Hostname: $(hostname)"
echo "IP: $(hostname -I 2>/dev/null)"

if ping -c 1 127.0.0.1 > /dev/null 2>&1; then
    echo "Netzwerkprüfung erfolgreich"
    echo "$(date): Netzwerkprüfung erfolgreich" >> "$LOG"
else
    echo "Netzwerkprüfung fehlgeschlagen"
    echo "$(date): Netzwerkprüfung fehlgeschlagen" >> "$LOG"
    exit 1
fi

if test -f /tmp/linux_block2/data/check.txt; then
    echo "Konfigurationsdatei vorhanden"
    echo "$(date): Konfigurationsdatei vorhanden" >> "$LOG"
else
    echo "Konfigurationsdatei fehlt"
    exit 1
fi

echo "Lokaler Musterablauf abgeschlossen"
EOF

chmod +x "$SCRIPT"
"$SCRIPT"

echo
echo "Log:"
cat "$LOG"

# SSH- und systemctl-Schritte werden in der realen Schulungsumgebung ergänzt.
